# DeepSeek-V4: Compressed Sparse Attention (CSA)

This notebook simplifies `Compressor` and `Indexer` to illustrate the core CSA data flow.

CSA has 3 main components:

- **Attention**: manages kv_cache, window_kv, and calls sparse attention
- **Compressor**: produces compressed KV and updates the attention kv_cache
- **Indexer**: a lightweight branch that scores compressed KV to select block-wise sparse indices

Indexer internally contains its own Compressor (with smaller head_dim) to produce compressed KV for scoring.

The pseudo-code for CSA:

```python
class Attention(nn.Module):
    def __init__(self, args):
        self.kv_cache = torch.register_buffer(...)       # window + compressed KV
        self.compressor = Compressor(args, ratio)        # outside Compressor, outside kv_cache
        self.indexer = Indexer(args, ratio)              # inside Compressor, inside kv_cache

    def forward(self, X):
        ckv = self.compressor(X)
        topk_idxs = self.indexer(X)                     # sparse selection on compressed KV
        kv = cat(kv, ckv)                                # concat window + compressed KV
        topk_idxs = cat(topk_idxs, window_idxs)          # merge sparse + window indices
        # prefill
        sparse_attn(q, kv, topk_idxs)
        # decoding
        sparse_attn(q, self.kv_cache, topk_idxs)
```

# Configuration

In [1]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [2]:
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

@dataclass
class ModelArgs:
    max_batch_size: int = 4
    max_seq_len: int = 4096
    vocab_size: int = 129280
    dim: int = 4096
    moe_inter_dim: int = 4096
    n_layers: int = 7
    n_heads: int = 64
    
    # mqa
    q_lora_rank: int = 1024
    head_dim: int = 512
    rope_head_dim: int = 64
    norm_eps: float = 1e-6
    o_groups: int = 8
    o_lora_rank: int = 1024
    window_size: int = 128
    compress_ratios: Tuple[int] = (0, 0, 4, 128, 4, 128, 4, 0)
    n_heads: int = 128
    
    # indexer
    index_n_heads: int = 64
    index_head_dim: int = 128
    index_topk: int = 512

args = ModelArgs()
args.index_topk = 3

# Input data
bsz = 2
seq_len = 100
X = torch.randn(bsz, seq_len, args.dim)

# Compressed Sparse Attention

## Simplified Compressor

The Compressor reduces the sequence length by `ratio`. In this simplified version, it produces random compressed KV vectors to illustrate the shape transformation.

Key point: ratio=4 (CSA) produces `L // 4` compressed vectors from an input of length `L`.

In [3]:
class Compressor(nn.Module):
    """Simplified Compressor: reduces sequence length by ratio (no learnable params)."""
    def __init__(self, args: ModelArgs, compress_ratio: int = 4, head_dim: int = 512):
        super().__init__()
        self.ratio = compress_ratio
        self.head_dim = head_dim
        self.kv_cache = None

    def forward(self, x, start_pos=0, debug=False):
        B, L, D = x.shape
        L_ckv = L // self.ratio
        cKV = torch.randn(B, L_ckv, self.head_dim)
        if self.kv_cache is not None and not debug:
            self.kv_cache[:bsz, :L_ckv] = cKV
        return cKV

In [4]:
# Demonstrate: 100 tokens compressed to 25 with ratio=4
compressor = Compressor(args, compress_ratio=4, head_dim=args.head_dim)
cKV = compressor(X, debug=True)
print(f"Input shape: {list(X.shape)}, Compressed KV shape: {list(cKV.shape)}")
print(f"Compression: {seq_len} tokens -> {seq_len // 4} compressed vectors (ratio=4)")

Input shape: [2, 100, 4096], Compressed KV shape: [2, 25, 512]
Compression: 100 tokens -> 25 compressed vectors (ratio=4)


## Simplified Indexer

The Indexer scores compressed KV to produce top-k indices for sparse selection.

Key point: it uses random scores (simplified) instead of learned multi-head scoring. The output shape is `[B, L, topk]` — each query selects `topk` compressed KV blocks.

In [5]:
class Indexer(nn.Module):
    """Simplified Indexer: uses random scores to select top-k compressed KV indices."""
    def __init__(self, args: ModelArgs, compress_ratio: int = 4):
        super().__init__()
        self.ratio = compress_ratio
        self.topk = args.index_topk
        self.head_dim = args.index_head_dim
        self.register_buffer(
            "kv_cache",
            torch.zeros(args.max_batch_size,
                        args.max_seq_len // compress_ratio,
                        self.head_dim),
            persistent=False
        )
        self.compressor = Compressor(args, compress_ratio=self.ratio, head_dim=self.head_dim)

    def forward(self, x: torch.Tensor, qr: torch.Tensor, start_pos: int = 0, offset: int = 128):
        B, L, D = x.size()
        L_ckv = L // self.ratio
        cKV = self.compressor(x)  # inner Compressor for scoring
        score = torch.randn(B, L, L_ckv)  # simplified: random scores
        topk_idxs = score.topk(k=self.topk, dim=-1)[1]
        return topk_idxs

In [6]:
# Each query selects top-k=3 compressed KV blocks
indexer = Indexer(args)

x = torch.randn(bsz, seq_len, args.dim)
qr = torch.randn(bsz, seq_len, args.index_head_dim)

topk_idxs = indexer(x, qr)
print(f"Input: {list(x.shape)}, Indexer output (topk_idxs): {list(topk_idxs.shape)}")
print(f"Per-query selection: each of {seq_len} queries picks {args.index_topk} compressed KV blocks")
print(f"Example topk indices for first batch, first 5 queries:\n{topk_idxs[0][:5]}")

Input: [2, 100, 4096], Indexer output (topk_idxs): [2, 100, 3]
Per-query selection: each of 100 queries picks 3 compressed KV blocks
Example topk indices for first batch, first 5 queries:
tensor([[19, 10, 18],
        [22,  4, 21],
        [19,  4, 23],
        [16,  6,  0],
        [ 0, 15, 21]])


## Full CSA Assembly

CSA combines: window KV + compressed KV (selected by Indexer top-k). The kv_cache layout is `[window_size + max_seq_len // ratio]`.

Key data flow:
1. `qr = wq_a(x)` -> low-rank query representation
2. `q = wq_b(qr)` -> multi-head query
3. `kv = wkv(x)` -> single-head KV (MQA)
4. `ckv = compressor(x)` -> compressed KV
5. `topk_idx = indexer(x, qr)` -> sparse selection indices (from inner compressed KV)
6. Merge window indices + topk indices
7. Update kv_cache with window KV and compressed KV
8. Call sparse_attn

In [7]:
def sparse_attn(q, kv, topk_idxs):
    """Placeholder: returns q unchanged (simplified)."""
    return q

class CompressedSparseAttention(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.ratio = 4
        self.indexer = Indexer(args, compress_ratio=self.ratio)
        self.head_dim = args.head_dim
        self.n_heads = args.n_heads
        self.dim = args.dim

        # kv_cache: [window_size + max_seq_len // ratio] per slot
        kv_cache_size = args.window_size + (args.max_seq_len // self.ratio if self.ratio else 0)
        self.register_buffer(
            "kv_cache",
            torch.zeros(args.max_batch_size,
                        kv_cache_size,
                        self.head_dim),
            persistent=False
        )
        self.compressor = Compressor(args, compress_ratio=self.ratio)

        q_lora_rank = args.q_lora_rank
        self.wq_a = nn.Linear(self.dim, q_lora_rank)
        self.wq_b = nn.Linear(q_lora_rank, self.n_heads * self.head_dim)
        self.wkv = nn.Linear(self.dim, self.head_dim)

        self.win_size = args.window_size

    def forward(self, x, start_pos=0):
        B, L, _ = x.shape
        ckv_len = L // self.ratio

        # Step 1-2: Query projection (low-rank -> multi-head)
        qr = self.wq_a(x)
        q = self.wq_b(qr)
        # Step 3: KV projection (single-head MQA)
        kv = self.wkv(x)
        # Step 4: Compressed KV
        ckv = self.compressor(x, start_pos)

        # Step 5: Indexer selects top-k compressed KV indices
        topk_idx = self.indexer(x, qr, start_pos, offset=self.win_size)
        # Step 6: Merge window indices + topk indices
        win_idx = torch.arange(0, self.win_size)
        win_idx = win_idx[None, None, :].repeat([B, L, 1])
        topk_idx = torch.cat([win_idx, topk_idx], dim=2)

        # Step 7: Update kv_cache
        win_size = min(self.win_size, L)
        self.kv_cache[:B, :win_size] = kv[:, -win_size:]
        self.kv_cache[:B, self.win_size:self.win_size + ckv_len] = ckv

        if start_pos == 0:  # prefill
            kv = torch.cat([kv, ckv], dim=1)
            o = sparse_attn(q, kv, topk_idx)
        else:  # decoding
            o = sparse_attn(q, self.kv_cache, topk_idx)

        return o

In [8]:
attention = CompressedSparseAttention(args)
print(attention)

CompressedSparseAttention(
  (indexer): Indexer(
    (compressor): Compressor()
  )
  (compressor): Compressor()
  (wq_a): Linear(in_features=4096, out_features=1024, bias=True)
  (wq_b): Linear(in_features=1024, out_features=65536, bias=True)
  (wkv): Linear(in_features=4096, out_features=512, bias=True)
)


In [9]:
# Forward pass: input [2, 100, 4096] -> output [2, 100, 65536]
x = torch.randn(bsz, seq_len, args.dim)
o = attention(x)
print(f"Input: {list(x.shape)}, Output: {list(o.shape)}")
print(f"Output dim = n_heads * head_dim = {args.n_heads} * {args.head_dim} = {args.n_heads * args.head_dim}")

Input: [2, 100, 4096], Output: [2, 100, 65536]
Output dim = n_heads * head_dim = 128 * 512 = 65536


# Analysis

1. Compared to HCA, the attention kv_cache size is the same. CSA adds Indexer's inner kv_cache and state.
2. CSA is a block-wise sparse pattern: sparsity operates on compressed KV blocks, not individual tokens.
3. Indexer is lightweight due to smaller parameters (`index_n_heads=64`, `index_head_dim=128`) and compressed KV (shorter sequence).

# Further Reading

- Indexer details: `./Indexer.ipynb`
- Sparse attention kernel analysis: `./lc5/`